# NB-EXP-001 · Cinematic Feature Extraction

**Cinematic Emotion Lab** · Bernard G.

---

## Experiment Overview

| Field | Value |
|-------|-------|
| **Notebook ID** | NB-EXP-001 |
| **Date** | 2026-06-06 |
| **Status** | Active |
| **Author** | Bernard G. — Film Composer · AI Architect |
| **Relates To** | EXP-001 baseline feature extraction |

---

## Research Question

> *Can low-level audio features extracted from cinematic music reliably encode emotional states such as tension, triumph, grief, and suspense?*

This notebook establishes the foundational feature extraction pipeline for all downstream ML experiments in the Cinematic Emotion Lab. We extract a comprehensive set of acoustic descriptors from raw audio clips and prepare a structured feature matrix for emotion classification tasks.

---

## Features Extracted

| Feature | Domain | Emotional Relevance |
|---------|--------|--------------------|
| Tempo (BPM) | Rhythmic | Arousal, energy, urgency |
| Spectral Centroid | Spectral | Brightness, tension |
| Chroma Features | Harmonic | Key, mode, harmonic color |
| RMS Energy | Amplitude | Loudness, dramatic weight |
| Zero Crossing Rate | Temporal | Noisiness, percussive density |
| MFCCs (13 coefficients) | Timbral | Timbre, instrumental texture |
| Harmonic/Percussive Ratio | Structural | Balance, orchestral texture |

---

## Folder Assumptions

```
cinematic-emotion-lab/
├── audio-samples/raw/          ← Place .wav / .mp3 / .flac clips here
├── audio-samples/features/     ← Extracted feature files (.csv, .npy)
└── visualizations/plots/       ← Generated figures saved here
```

Audio files must be placed in `../../audio-samples/raw/` relative to this notebook.
A synthetic demo clip is generated automatically if no files are present.


---

## 0 · Environment Setup

Install or verify all required libraries. This cell is safe to re-run.

In [ ]:
# Uncomment to install dependencies in a fresh environment
# !pip install librosa matplotlib seaborn pandas numpy plotly soundfile scipy kaleido

## 1 · Imports and Global Configuration

In [ ]:
import os
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import scipy.signal

warnings.filterwarnings('ignore')

# ── Cinematic color palette ───────────────────────────────────────────────────
PALETTE = {
    'bg':        '#0D0D0F',
    'surface':   '#1A1A2E',
    'accent1':   '#E94560',   # crimson — tension / drama
    'accent2':   '#F5A623',   # amber   — triumph / warmth
    'accent3':   '#4FC3F7',   # ice blue — suspense / cold
    'accent4':   '#A8E6CF',   # sage    — resolution / peace
    'text':      '#E8E8E8',
    'muted':     '#666680',
}

COLORS = [PALETTE['accent1'], PALETTE['accent2'], PALETTE['accent3'], PALETTE['accent4']]

# ── Matplotlib cinematic theme ────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  PALETTE['bg'],
    'axes.facecolor':    PALETTE['surface'],
    'axes.edgecolor':    PALETTE['muted'],
    'axes.labelcolor':   PALETTE['text'],
    'axes.titlecolor':   PALETTE['text'],
    'xtick.color':       PALETTE['muted'],
    'ytick.color':       PALETTE['muted'],
    'text.color':        PALETTE['text'],
    'grid.color':        PALETTE['muted'],
    'grid.alpha':        0.2,
    'figure.titlesize':  16,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'font.family':       'monospace',
})

# ── Project paths ─────────────────────────────────────────────────────────────
NOTEBOOK_DIR   = Path('.')
PROJECT_ROOT   = NOTEBOOK_DIR.parent.parent
AUDIO_RAW_DIR  = PROJECT_ROOT / 'audio-samples' / 'raw'
FEATURES_DIR   = PROJECT_ROOT / 'audio-samples' / 'features'
PLOTS_DIR      = PROJECT_ROOT / 'visualizations' / 'plots'

for d in [AUDIO_RAW_DIR, FEATURES_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Audio config ──────────────────────────────────────────────────────────────
SAMPLE_RATE    = 22050   # Hz — standard for MIR
N_MFCC         = 13      # MFCC coefficients
HOP_LENGTH     = 512     # frames
N_FFT          = 2048    # FFT window size

print(f"Project root : {PROJECT_ROOT.resolve()}")
print(f"Audio input  : {AUDIO_RAW_DIR.resolve()}")
print(f"Feature output: {FEATURES_DIR.resolve()}")
print(f"Plots output : {PLOTS_DIR.resolve()}")

---

## 2 · Audio Loading

We scan the `audio-samples/raw/` directory for supported audio formats (`.wav`, `.mp3`, `.flac`, `.aif`, `.aiff`). 

If no files are found, a **synthetic demo clip** is generated — a sinusoidal composition with varying harmonic layers to simulate a cinematic passage. This ensures the notebook runs end-to-end without requiring audio files.

> For real experiments, place labeled audio clips in `audio-samples/raw/` following the naming convention:
> `FILMCODE_COMPOSER_SCENETYPE_EMOTION_BPMbpm.wav`

In [ ]:
SUPPORTED_FORMATS = {'.wav', '.mp3', '.flac', '.aif', '.aiff'}


def scan_audio_directory(directory: Path) -> List[Path]:
    """Return all supported audio files in directory."""
    files = sorted([
        f for f in directory.iterdir()
        if f.suffix.lower() in SUPPORTED_FORMATS
    ])
    return files


def generate_synthetic_clip(
    duration: float = 10.0,
    sr: int = SAMPLE_RATE,
    label: str = 'DEMO001_BG_synthetic_tension_85bpm'
) -> Tuple[np.ndarray, int, str]:
    """Generate a synthetic cinematic audio clip for demonstration."""
    t = np.linspace(0, duration, int(sr * duration))

    # Layered harmonics simulating strings + brass
    fundamental = 220.0  # A3
    signal = (
        0.5  * np.sin(2 * np.pi * fundamental * t) +
        0.25 * np.sin(2 * np.pi * fundamental * 2 * t) +
        0.15 * np.sin(2 * np.pi * fundamental * 3 * t) +
        0.10 * np.sin(2 * np.pi * fundamental * 4 * t)
    )

    # Amplitude envelope — swell then release
    envelope = np.exp(-0.3 * t) * (1 - np.exp(-2 * t))
    signal = (signal * envelope).astype(np.float32)

    # Add light noise for realism
    signal += 0.01 * np.random.randn(len(signal)).astype(np.float32)

    save_path = AUDIO_RAW_DIR / f"{label}.wav"
    import soundfile as sf
    sf.write(save_path, signal, sr)
    print(f"  Synthetic clip saved → {save_path.name}")
    return signal, sr, label


def load_audio_clips(directory: Path) -> List[Dict]:
    """Load all audio clips from directory. Returns list of clip dicts."""
    files = scan_audio_directory(directory)

    if not files:
        print("No audio files found — generating synthetic demo clip...")
        y, sr, label = generate_synthetic_clip()
        files = scan_audio_directory(directory)

    clips = []
    for path in files:
        try:
            y, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
            clips.append({
                'path':     path,
                'filename': path.stem,
                'y':        y,
                'sr':       sr,
                'duration': librosa.get_duration(y=y, sr=sr),
            })
            print(f"  ✓ Loaded: {path.name} ({clips[-1]['duration']:.2f}s)")
        except Exception as e:
            print(f"  ✗ Failed: {path.name} — {e}")

    print(f"\nTotal clips loaded: {len(clips)}")
    return clips


clips = load_audio_clips(AUDIO_RAW_DIR)

---

## 3 · Feature Extraction

We extract seven categories of acoustic features. Each captures a distinct dimension of musical and emotional expression:

| Feature | Library Call | Emotional Dimension |
|---------|-------------|--------------------|
| **Tempo** | `librosa.beat.beat_track` | Pacing, urgency, energy |
| **Spectral Centroid** | `librosa.feature.spectral_centroid` | Brightness — higher = sharper/tenser |
| **Chroma** | `librosa.feature.chroma_cqt` | Harmonic content, key, mode |
| **RMS Energy** | `librosa.feature.rms` | Loudness, dynamic range, weight |
| **Zero Crossing Rate** | `librosa.feature.zero_crossing_rate` | Noisiness, percussive texture |
| **MFCCs** | `librosa.feature.mfcc` | Timbral identity, instrument blend |
| **H/P Separation** | `librosa.effects.hpss` | Harmonic vs. percussive balance |

All time-varying features are summarized as **mean** and **standard deviation** across frames, yielding a fixed-length feature vector per clip suitable for ML.

In [ ]:
def extract_features(clip: Dict) -> Dict:
    """Extract full acoustic feature set from a single audio clip."""
    y, sr = clip['y'], clip['sr']
    features = {'filename': clip['filename'], 'duration_s': clip['duration']}

    # ── 1. Tempo ──────────────────────────────────────────────────────────────
    tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP_LENGTH)
    features['tempo_bpm'] = float(tempo)

    # ── 2. Spectral Centroid ──────────────────────────────────────────────────
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features['spectral_centroid_mean'] = float(np.mean(centroid))
    features['spectral_centroid_std']  = float(np.std(centroid))

    # ── 3. Spectral Bandwidth ─────────────────────────────────────────────────
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features['spectral_bandwidth_mean'] = float(np.mean(bandwidth))
    features['spectral_bandwidth_std']  = float(np.std(bandwidth))

    # ── 4. Spectral Rolloff ───────────────────────────────────────────────────
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features['spectral_rolloff_mean'] = float(np.mean(rolloff))
    features['spectral_rolloff_std']  = float(np.std(rolloff))

    # ── 5. Chroma Features ────────────────────────────────────────────────────
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=HOP_LENGTH)
    pitch_classes = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    for i, pc in enumerate(pitch_classes):
        features[f'chroma_{pc}_mean'] = float(np.mean(chroma[i]))
        features[f'chroma_{pc}_std']  = float(np.std(chroma[i]))
    features['chroma_mean_overall'] = float(np.mean(chroma))
    features['dominant_pitch_class'] = pitch_classes[int(np.argmax([np.mean(chroma[i]) for i in range(12)]))]

    # ── 6. RMS Energy ─────────────────────────────────────────────────────────
    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)
    features['rms_mean']    = float(np.mean(rms))
    features['rms_std']     = float(np.std(rms))
    features['rms_max']     = float(np.max(rms))
    features['dynamic_range_db'] = float(20 * np.log10(np.max(rms) / (np.mean(rms) + 1e-9)))

    # ── 7. Zero Crossing Rate ─────────────────────────────────────────────────
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)
    features['zcr_mean'] = float(np.mean(zcr))
    features['zcr_std']  = float(np.std(zcr))

    # ── 8. MFCCs ─────────────────────────────────────────────────────────────
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    for i in range(N_MFCC):
        features[f'mfcc_{i+1:02d}_mean'] = float(np.mean(mfccs[i]))
        features[f'mfcc_{i+1:02d}_std']  = float(np.std(mfccs[i]))

    # ── 9. Harmonic / Percussive Separation ───────────────────────────────────
    y_harm, y_perc = librosa.effects.hpss(y)
    harm_energy = float(np.mean(y_harm ** 2))
    perc_energy = float(np.mean(y_perc ** 2))
    total_energy = harm_energy + perc_energy + 1e-9
    features['harmonic_ratio']   = harm_energy / total_energy
    features['percussive_ratio'] = perc_energy / total_energy
    features['hp_balance']       = harm_energy / (perc_energy + 1e-9)

    # Store raw arrays for visualization (not exported to CSV)
    clip['_centroid'] = centroid
    clip['_chroma']   = chroma
    clip['_rms']      = rms
    clip['_mfccs']    = mfccs
    clip['_zcr']      = zcr
    clip['_y_harm']   = y_harm
    clip['_y_perc']   = y_perc
    clip['_beat_frames'] = beat_frames

    return features


print("Extracting features...")
all_features = []
for clip in clips:
    feats = extract_features(clip)
    all_features.append(feats)
    print(f"  ✓ {clip['filename']} — tempo={feats['tempo_bpm']:.1f} BPM | "
          f"centroid={feats['spectral_centroid_mean']:.0f} Hz | "
          f"dominant_key={feats['dominant_pitch_class']}")

features_df = pd.DataFrame(all_features)
print(f"\nFeature matrix: {features_df.shape[0]} clips × {features_df.shape[1]} features")
features_df.head()

---

## 4 · Composer Annotation Placeholder

This section provides the annotation schema for composer-labeled emotional data. Labels will be collected via the annotation pipeline and merged with extracted features for supervised learning.

> **Note for annotators:** Use your compositional intuition. Describe what the music *communicates*, not just its technical properties.

In [ ]:
EMOTION_LABELS = ['tension', 'triumph', 'grief', 'suspense', 'joy', 'dread', 'longing', 'ambiguity']
INTENSITY_SCALE = [1, 2, 3, 4, 5]  # 1=minimal, 5=maximal

def build_annotation_template(filenames: List[str]) -> pd.DataFrame:
    """Build empty annotation template for composer labeling."""
    rows = []
    for fn in filenames:
        row = {
            'filename':              fn,
            'primary_emotion':       None,   # One of EMOTION_LABELS
            'secondary_emotion':     None,   # Optional second label
            'emotional_intensity':   None,   # 1–5
            'narrative_position':    None,   # 'opening','rising','climax','falling','resolution'
            'harmonic_mode':         None,   # 'major','minor','modal','atonal'
            'orchestration_density': None,   # 1–5
            'composer_notes':        None,   # Free text
        }
        rows.append(row)
    return pd.DataFrame(rows)


annotation_df = build_annotation_template([c['filename'] for c in clips])

# ── DEMO: Inject a mock annotation for the synthetic clip ─────────────────────
if len(annotation_df) > 0:
    annotation_df.loc[0, 'primary_emotion']       = 'tension'
    annotation_df.loc[0, 'secondary_emotion']     = 'suspense'
    annotation_df.loc[0, 'emotional_intensity']   = 4
    annotation_df.loc[0, 'narrative_position']    = 'rising'
    annotation_df.loc[0, 'harmonic_mode']         = 'minor'
    annotation_df.loc[0, 'orchestration_density'] = 3
    annotation_df.loc[0, 'composer_notes']        = (
        'Descending minor lines over sustained brass pedal. '
        'Spectral density builds toward unresolved dominant.'
    )

print("Annotation template:")
annotation_df

---

## 5 · Waveform & Spectrogram Visualizations

For each audio clip we generate a **four-panel cinematic analysis figure**:

1. **Waveform** — amplitude over time with beat markers
2. **Mel Spectrogram** — time-frequency energy distribution
3. **Harmonic / Percussive** — separated waveforms overlaid
4. **Chromagram** — pitch-class energy over time

In [ ]:
def plot_clip_analysis(clip: Dict, annotation: Optional[pd.Series] = None) -> plt.Figure:
    """Generate four-panel cinematic analysis figure for a single clip."""
    y, sr = clip['y'], clip['sr']
    filename = clip['filename']
    label = annotation['primary_emotion'] if annotation is not None and annotation['primary_emotion'] else 'unlabeled'

    fig = plt.figure(figsize=(18, 12), facecolor=PALETTE['bg'])
    fig.suptitle(
        f'Cinematic Audio Analysis  ·  {filename}\nEmotion: {label.upper()}',
        fontsize=15, fontweight='bold', color=PALETTE['text'], y=0.98
    )

    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.3)

    # ── Panel 1: Waveform ─────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    times = np.linspace(0, clip['duration'], len(y))
    ax1.plot(times, y, color=PALETTE['accent1'], linewidth=0.5, alpha=0.9)
    ax1.fill_between(times, y, alpha=0.15, color=PALETTE['accent1'])

    if '_beat_frames' in clip and len(clip['_beat_frames']) > 0:
        beat_times = librosa.frames_to_time(clip['_beat_frames'], sr=sr, hop_length=HOP_LENGTH)
        for bt in beat_times:
            ax1.axvline(x=bt, color=PALETTE['accent2'], alpha=0.4, linewidth=0.6)

    ax1.set_title('Waveform  +  Beat Grid', fontweight='bold')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Amplitude')
    ax1.grid(True, alpha=0.15)

    # ── Panel 2: Mel Spectrogram ──────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(
        S_db, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel',
        ax=ax2, cmap='magma'
    )
    plt.colorbar(img, ax=ax2, format='%+2.0f dB', label='dB')
    ax2.set_title('Mel Spectrogram', fontweight='bold')

    # ── Panel 3: Harmonic / Percussive Separation ─────────────────────────────
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(times, clip['_y_harm'], color=PALETTE['accent3'], linewidth=0.5, alpha=0.85, label='Harmonic')
    ax3.plot(times, clip['_y_perc'], color=PALETTE['accent2'], linewidth=0.5, alpha=0.65, label='Percussive')
    ax3.set_title('Harmonic / Percussive Separation', fontweight='bold')
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Amplitude')
    ax3.legend(loc='upper right', fontsize=9, framealpha=0.3)
    ax3.grid(True, alpha=0.15)

    # ── Panel 4: Chromagram ───────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 1])
    pitch_classes = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    chroma_img = librosa.display.specshow(
        clip['_chroma'], sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='chroma',
        ax=ax4, cmap='YlOrRd'
    )
    plt.colorbar(chroma_img, ax=ax4, label='Intensity')
    ax4.set_title('Chromagram  (Pitch-Class Energy)', fontweight='bold')

    # Footer
    fig.text(0.5, 0.01,
        'Cinematic Emotion Lab · Bernard G. · NB-EXP-001',
        ha='center', fontsize=9, color=PALETTE['muted']
    )

    save_path = PLOTS_DIR / f"NB-EXP-001_{filename}_analysis.png"
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=PALETTE['bg'])
    print(f"  Figure saved → {save_path.name}")
    return fig


for i, clip in enumerate(clips):
    ann = annotation_df.iloc[i] if i < len(annotation_df) else None
    fig = plot_clip_analysis(clip, ann)
    plt.show()

---

## 6 · Feature Charts

### 6.1 · MFCC Heatmap

The MFCC heatmap shows how timbral character evolves over time. Coefficient 1 (C1) primarily encodes loudness; C2–C13 capture progressively finer timbral detail. Consistent patterns indicate stable orchestration; irregular patterns suggest transitions or emotional shifts.

In [ ]:
def plot_mfcc_heatmap(clip: Dict) -> plt.Figure:
    """Render MFCC coefficient heatmap over time."""
    fig, ax = plt.subplots(figsize=(16, 5), facecolor=PALETTE['bg'])

    img = librosa.display.specshow(
        clip['_mfccs'], sr=clip['sr'], hop_length=HOP_LENGTH,
        x_axis='time', ax=ax, cmap='coolwarm'
    )
    plt.colorbar(img, ax=ax, label='MFCC Coefficient Value')
    ax.set_yticks(range(N_MFCC))
    ax.set_yticklabels([f'C{i+1}' for i in range(N_MFCC)], fontsize=9)
    ax.set_title(f'MFCC Heatmap  ·  {clip["filename"]}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('MFCC Coefficient')

    fig.text(0.5, -0.04,
        'Cinematic Emotion Lab · Bernard G. · NB-EXP-001',
        ha='center', fontsize=9, color=PALETTE['muted']
    )

    save_path = PLOTS_DIR / f"NB-EXP-001_{clip['filename']}_mfcc-heatmap.png"
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=PALETTE['bg'])
    print(f"  Figure saved → {save_path.name}")
    return fig


for clip in clips:
    fig = plot_mfcc_heatmap(clip)
    plt.show()

### 6.2 · RMS Energy & Spectral Centroid Over Time

These two features in tandem reveal the **emotional arc** of a clip:
- **RMS Energy** tracks dramatic weight and loudness over time
- **Spectral Centroid** tracks brightness — rising centroid often signals increasing tension or brightness

In [ ]:
def plot_energy_centroid(clip: Dict) -> plt.Figure:
    """Dual-axis plot of RMS energy and spectral centroid over time."""
    frames = clip['_rms'].shape[1]
    frame_times = librosa.frames_to_time(np.arange(frames), sr=clip['sr'], hop_length=HOP_LENGTH)

    fig, ax1 = plt.subplots(figsize=(16, 5), facecolor=PALETTE['bg'])
    ax2 = ax1.twinx()
    ax2.set_facecolor(PALETTE['surface'])

    rms_vals      = clip['_rms'][0]
    centroid_vals = clip['_centroid'][0][:frames]

    ax1.fill_between(frame_times, rms_vals, alpha=0.35, color=PALETTE['accent1'])
    ax1.plot(frame_times, rms_vals, color=PALETTE['accent1'], linewidth=1.5, label='RMS Energy')
    ax2.plot(frame_times, centroid_vals, color=PALETTE['accent3'], linewidth=1.5,
             linestyle='--', label='Spectral Centroid')

    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('RMS Energy', color=PALETTE['accent1'])
    ax2.set_ylabel('Spectral Centroid (Hz)', color=PALETTE['accent3'])
    ax1.tick_params(axis='y', labelcolor=PALETTE['accent1'])
    ax2.tick_params(axis='y', labelcolor=PALETTE['accent3'])

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10, framealpha=0.3)

    ax1.set_title(f'RMS Energy  &  Spectral Centroid  ·  {clip["filename"]}',
                  fontweight='bold', fontsize=13)
    ax1.grid(True, alpha=0.15)

    fig.text(0.5, -0.04,
        'Cinematic Emotion Lab · Bernard G. · NB-EXP-001',
        ha='center', fontsize=9, color=PALETTE['muted']
    )

    save_path = PLOTS_DIR / f"NB-EXP-001_{clip['filename']}_energy-centroid.png"
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=PALETTE['bg'])
    print(f"  Figure saved → {save_path.name}")
    return fig


for clip in clips:
    fig = plot_energy_centroid(clip)
    plt.show()

### 6.3 · Harmonic vs. Percussive Balance — Interactive Plotly

In [ ]:
def plot_hp_balance_plotly(df: pd.DataFrame) -> go.Figure:
    """Interactive bar chart of harmonic/percussive balance per clip."""
    fig = go.Figure()

    fig.add_trace(go.Bar(
        name='Harmonic',
        x=df['filename'],
        y=df['harmonic_ratio'],
        marker_color=PALETTE['accent3'],
        opacity=0.85,
    ))
    fig.add_trace(go.Bar(
        name='Percussive',
        x=df['filename'],
        y=df['percussive_ratio'],
        marker_color=PALETTE['accent1'],
        opacity=0.85,
    ))

    fig.update_layout(
        barmode='stack',
        title=dict(text='Harmonic vs. Percussive Energy Balance', font=dict(size=16, color=PALETTE['text'])),
        xaxis=dict(title='Clip', color=PALETTE['text'], tickfont=dict(color=PALETTE['muted'])),
        yaxis=dict(title='Energy Ratio', color=PALETTE['text'], tickfont=dict(color=PALETTE['muted'])),
        plot_bgcolor=PALETTE['surface'],
        paper_bgcolor=PALETTE['bg'],
        font=dict(color=PALETTE['text']),
        legend=dict(font=dict(color=PALETTE['text'])),
        height=420,
    )

    save_path = PLOTS_DIR / 'NB-EXP-001_hp-balance.html'
    fig.write_html(str(save_path))
    print(f"  Interactive chart saved → {save_path.name}")
    return fig


fig_hp = plot_hp_balance_plotly(features_df)
fig_hp.show()

### 6.4 · Chroma Profile — Interactive Radar Chart

In [ ]:
def plot_chroma_radar_plotly(df: pd.DataFrame) -> go.Figure:
    """Polar radar chart of mean chroma energy per pitch class."""
    pitch_classes = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    fig = go.Figure()

    for i, row in df.iterrows():
        values = [row[f'chroma_{pc}_mean'] for pc in pitch_classes]
        values_closed = values + [values[0]]  # close the loop
        pcs_closed    = pitch_classes + [pitch_classes[0]]

        fig.add_trace(go.Scatterpolar(
            r=values_closed,
            theta=pcs_closed,
            fill='toself',
            name=row['filename'][:30],
            opacity=0.7,
            line=dict(color=COLORS[i % len(COLORS)], width=2),
        ))

    fig.update_layout(
        title=dict(text='Chroma Profile  (Pitch-Class Energy Distribution)', font=dict(size=16, color=PALETTE['text'])),
        polar=dict(
            bgcolor=PALETTE['surface'],
            radialaxis=dict(visible=True, color=PALETTE['muted']),
            angularaxis=dict(color=PALETTE['text'], tickfont=dict(size=12)),
        ),
        paper_bgcolor=PALETTE['bg'],
        font=dict(color=PALETTE['text']),
        legend=dict(font=dict(color=PALETTE['text'])),
        height=500,
    )

    save_path = PLOTS_DIR / 'NB-EXP-001_chroma-radar.html'
    fig.write_html(str(save_path))
    print(f"  Interactive chart saved → {save_path.name}")
    return fig


fig_chroma = plot_chroma_radar_plotly(features_df)
fig_chroma.show()

### 6.5 · MFCC Summary — Interactive Bar Chart

In [ ]:
def plot_mfcc_summary_plotly(df: pd.DataFrame) -> go.Figure:
    """Interactive grouped bar chart of mean MFCC values per clip."""
    mfcc_cols = [f'mfcc_{i+1:02d}_mean' for i in range(N_MFCC)]
    mfcc_labels = [f'C{i+1}' for i in range(N_MFCC)]

    fig = go.Figure()
    for i, row in df.iterrows():
        vals = [row[col] for col in mfcc_cols]
        fig.add_trace(go.Bar(
            name=row['filename'][:30],
            x=mfcc_labels,
            y=vals,
            marker_color=COLORS[i % len(COLORS)],
            opacity=0.8,
        ))

    fig.update_layout(
        barmode='group',
        title=dict(text='MFCC Coefficient Means  (Timbral Profile per Clip)', font=dict(size=16, color=PALETTE['text'])),
        xaxis=dict(title='MFCC Coefficient', color=PALETTE['text']),
        yaxis=dict(title='Mean Value', color=PALETTE['text']),
        plot_bgcolor=PALETTE['surface'],
        paper_bgcolor=PALETTE['bg'],
        font=dict(color=PALETTE['text']),
        legend=dict(font=dict(color=PALETTE['text'])),
        height=420,
    )

    save_path = PLOTS_DIR / 'NB-EXP-001_mfcc-summary.html'
    fig.write_html(str(save_path))
    print(f"  Interactive chart saved → {save_path.name}")
    return fig


fig_mfcc = plot_mfcc_summary_plotly(features_df)
fig_mfcc.show()

---

## 7 · Export Feature Matrix to CSV

The extracted feature matrix is merged with composer annotations and exported to `audio-samples/features/`. This CSV becomes the primary input for all downstream ML experiments.

**Output file:** `EXP-001_features_combined_YYYY-MM-DD.csv`

In [ ]:
from datetime import date

today = date.today().strftime('%Y-%m-%d')

# Merge features with annotations
annotation_export = annotation_df.drop(columns=['filename'])  # avoid duplicate
combined_df = pd.concat([features_df, annotation_export], axis=1)

# Export
output_path = FEATURES_DIR / f'EXP-001_features_combined_{today}.csv'
combined_df.to_csv(output_path, index=False)
print(f"Feature matrix exported → {output_path}")
print(f"Shape: {combined_df.shape[0]} clips × {combined_df.shape[1]} columns")
print(f"\nColumn groups:")
print(f"  Audio metadata      : {len([c for c in combined_df.columns if c in ['filename','duration_s']])}")
print(f"  Tempo features      : {len([c for c in combined_df.columns if 'tempo' in c])}")
print(f"  Spectral features   : {len([c for c in combined_df.columns if 'spectral' in c or 'rolloff' in c or 'bandwidth' in c])}")
print(f"  Chroma features     : {len([c for c in combined_df.columns if 'chroma' in c or 'dominant' in c])}")
print(f"  Energy features     : {len([c for c in combined_df.columns if 'rms' in c or 'dynamic' in c])}")
print(f"  ZCR features        : {len([c for c in combined_df.columns if 'zcr' in c])}")
print(f"  MFCC features       : {len([c for c in combined_df.columns if 'mfcc' in c])}")
print(f"  H/P features        : {len([c for c in combined_df.columns if 'harmonic' in c or 'percussive' in c or 'hp_' in c])}")
print(f"  Annotation columns  : {len([c for c in combined_df.columns if c in annotation_df.columns and c != 'filename'])}")
combined_df.head()

---

## 8 · Feature Summary Statistics

In [ ]:
summary_cols = [
    'tempo_bpm', 'spectral_centroid_mean', 'rms_mean',
    'zcr_mean', 'harmonic_ratio', 'percussive_ratio',
    'dynamic_range_db', 'hp_balance'
]

print("=" * 60)
print("  FEATURE SUMMARY — Cinematic Emotion Lab · EXP-001")
print("=" * 60)
display(features_df[summary_cols].describe().round(4))

---

## 9 · Results & Observations

> **Fill this section after running the notebook on real clips.**

### Key Observations

- `[ ]` Tempo range across clips:
- `[ ]` Dominant pitch classes found:
- `[ ]` Harmonic vs. percussive balance patterns:
- `[ ]` MFCC coefficient most variable across clips:
- `[ ]` Spectral centroid correlation with annotated emotion:

### Hypotheses for Next Experiment

1. Clips labeled `tension` will show higher spectral centroid and lower harmonic ratio
2. Clips labeled `grief` will show lower tempo and higher MFCC-C1 values
3. RMS dynamic range will correlate positively with `emotional_intensity`

---

## 10 · Next Steps

| Step | Notebook | Status |
|------|----------|--------|
| Collect 20+ labeled cinematic clips | — | Pending |
| Complete composer annotation pass | — | Pending |
| Baseline SVM emotion classifier | `NB-EXP-002` | Planned |
| Dimensionality reduction (UMAP) of feature space | `NB-VIZ-001` | Planned |
| Correlation analysis: features vs. emotion labels | `NB-EDA-002` | Planned |

---

## Notebook Metadata

```
Lab       : Cinematic Emotion Lab
Notebook  : NB-EXP-001
Author    : Bernard G.
Date      : 2026-06-06
Version   : 1.0
Status    : Active
License   : MIT
```

*Cinematic Emotion Lab — Where film composition meets machine intelligence.*